In [1]:
import yaml, copy
from pathlib import Path

def resolve_project_root() -> Path:
    root = Path.cwd()
    if not (root / "configs").exists() and (root.parent / "configs").exists():
        root = root.parent
    return root

PROJECT = resolve_project_root()
with open(PROJECT / "configs/config.yaml") as f:
    BASE = yaml.safe_load(f)

In [2]:
EXPERIMENT_NAME = "ws_final"
CONFIGS_DIR = PROJECT / "configs" / "generated" / EXPERIMENT_NAME
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
import numpy as np

n_events_range = (10000, 90000, 10000)
batch_size_powers = (5, 11)

n_events_vals = list(range(*n_events_range[:2], n_events_range[2])) + [n_events_range[1]]
batch_size_vals = [2**i for i in range(batch_size_powers[0], batch_size_powers[1] + 1)]
sigma_vals = [0.008, 0.01, 0.02, 0.03]

In [5]:
for n_ev in n_events_vals:
    cfg = copy.deepcopy(BASE)
    cfg['aging']['n_events'] = n_ev
    path = CONFIGS_DIR / f"config_n{n_ev}.yaml"
    with open(path, 'w') as f:
        yaml.dump(cfg, f, sort_keys=False)

In [10]:
for bs in batch_size_vals:
    cfg = copy.deepcopy(BASE)
    cfg['wgan_params']['batch_size'] = bs
    path = CONFIGS_DIR / f"config_b{bs}.yaml"
    with open(path, 'w') as f:
        yaml.dump(cfg, f, sort_keys=False)

In [5]:
for sigma in sigma_vals:
    cfg = copy.deepcopy(BASE)
    cfg['aging']['add_noise'] = True
    cfg['aging']['sigma'] = float(sigma)
    sigma_str = str(sigma).replace(".", "p")
    path = CONFIGS_DIR / f"config_sigma{sigma_str}.yaml"
    with open(path, 'w') as f:
        yaml.dump(cfg, f, sort_keys=False)

In [ ]:
EXPERIMENT_NAME = "noise_sigma_sweep"
CONFIGS_DIR = PROJECT / "configs" / "generated" / EXPERIMENT_NAME
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)

sigma_vals = [0.008, 0.01, 0.02, 0.03]
clamp_methods = ["clip", "normalize", "truncate"]

for sigma in sigma_vals:
    for method in clamp_methods:
        cfg = copy.deepcopy(BASE)
        cfg["aging"]["add_noise"] = True
        cfg["aging"]["sigma"] = sigma
        cfg["aging"]["noise_clamp_method"] = method
        sigma_str = str(sigma).replace(".", "p")
        path = CONFIGS_DIR / f"config_sigma{sigma_str}_{method}.yaml"
        with open(path, "w") as f:
            yaml.dump(cfg, f, sort_keys=False)

print(f"Сгенерировано {len(sigma_vals) * len(clamp_methods)} конфигов в {CONFIGS_DIR}")